[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-processing/N2N-SPRVS/blob/main/N2N-SPRVS.ipynb)

# N2N-SPRVS-3D (Noise2Noise Structure Preserving Random Voxel) tomogram denoising

In [ ]:
REO_folder = "~/REO/empiar11419_stack_ali_crop/"
STD_DEV = 2.5 # 2.5
SPATIAL_SIZE = 5 # 5
SIGMA_K = 4.5 # 4.5
FILTER_SIZE = 9 #9

In [ ]:
gpu_id = 0

In [ ]:
N = 1  # >=1 Number of generated tomograms used for training the denoiser

In [ ]:
# %pip install numpy
import numpy as np

In [ ]:
sliice = np.s_[803:304, 659:160]

In [ ]:
!nvidia-smi

## Packages

In [ ]:
# %pip install cupy-cuda12x # Check the CUDA installed with nvcc --version (that should match with the version provided by nvidia-smi)
# %pip install numba
# %pip install tqdm ipywidgets
# %pip install scipy
# %pip install "optical_flow_3D @ git+https://github.com/vicente-gonzalez-ruiz/optical_flow_3D"
import optical_flow_3D

In [ ]:
#%pip install mrcfile
import mrcfile

In [ ]:
# %pip install numpy
import numpy as np

In [ ]:
# %pip install tqdm ipywidgets
from tqdm.notebook import tqdm

In [ ]:
# %pip install matplotlib
import matplotlib.pyplot as plt

In [ ]:
# %pip install gdown
import gdown

In [ ]:
from pathlib import Path

In [ ]:
import os

## Download a noisy tomogram

In [ ]:
url="https://drive.google.com/file/d/1VSWgd8xS4zlw6amzTdqz5OdsnGh6aoCo"
#gdown.download(url, output="000.mrc", quiet=False, use_cookies=False)

In [ ]:
X = mrcfile.open("000.mrc").data

In [ ]:
X.shape

## Generate another version of the tomogram

In [ ]:
estimator = optical_flow_3D.OF3D.Farneback3D(
  iters=5,
  num_levels=3,
  scale=0.5,
  spatial_size=5, #9
  sigma_k=4.5, # 1.5
  filter_type="gaussian",
  filter_size=9, #9
  presmoothing=None,
  device_id=0)

In [ ]:
estimator = optical_flow_3D.OF3D.Farneback3D(
  iters=5,
  num_levels=3,
  scale=0.5,
  spatial_size=SPATIAL_SIZE, #9
  sigma_k=SIGMA_K, # 1.5
  filter_type="gaussian",
  filter_size=FILTER_SIZE, #9
  presmoothing=None,
  device_id=gpu_id)

In [ ]:
import scipy.ndimage

def randomize(vol, mean=0.0, std_dev=1.0):
  depth, height, width = vol.shape[:3]
  z_coords, y_coords, x_coords = np.meshgrid(range(depth), range(height), range(width), indexing="ij")
  flattened_x_coords = x_coords.flatten()
  flattened_y_coords = y_coords.flatten()
  flattened_z_coords = z_coords.flatten()
  #print(np.max(flattened_z_coords), np.max(flattened_y_coords), np.max(flattened_x_coords))
  #print(flattened_x_coords.dtype)
  displacements_x = np.random.normal(mean, std_dev, flattened_x_coords.shape).astype(np.int32)
  displacements_y = np.random.normal(mean, std_dev, flattened_y_coords.shape).astype(np.int32)
  displacements_z = np.random.normal(mean, std_dev, flattened_z_coords.shape).astype(np.int32)
  #_d = 5
  #displacements_x = np.random.uniform(low=-_d, high=_d, size=flattened_x_coords.shape).astype(np.int32)
  #displacements_y = np.random.uniform(low=-_d, high=_d, size=flattened_y_coords.shape).astype(np.int32)
  #displacements_z = np.random.uniform(low=-_d, high=_d, size=flattened_z_coords.shape).astype(np.int32)
  print("min displacements", np.min(displacements_z), np.min(displacements_y), np.min(displacements_x))
  print("average abs(displacements)", np.average(np.abs(displacements_z)), np.average(np.abs(displacements_y)), np.average(np.abs(displacements_x)))
  print("max displacements", np.max(displacements_z), np.max(displacements_y), np.max(displacements_x))
  randomized_x_coords = flattened_x_coords + displacements_x
  randomized_y_coords = flattened_y_coords + displacements_y
  randomized_z_coords = flattened_z_coords + displacements_z
  #print("max displacements", np.max(randomized_z_coords), np.max(randomized_y_coords), np.max(randomized_x_coords))
  #randomized_x_coords = np.mod(randomized_x_coords, width)
  #randomized_y_coords = np.mod(randomized_y_coords, height)
  #randomized_z_coords = np.mod(randomized_z_coords, depth)
  randomized_x_coords = np.clip(randomized_x_coords, 0, width - 1) # Clip the randomized coordinates to stay within image bounds
  randomized_y_coords = np.clip(randomized_y_coords, 0, height - 1)
  randomized_z_coords = np.clip(randomized_z_coords, 0, depth - 1)
  #print(np.max(randomized_z_coords), np.max(randomized_y_coords), np.max(randomized_x_coords))
  #randomized_vol = np.ones_like(vol)*np.average(vol) #np.zeros_like(vol)
  randomized_vol = np.zeros_like(vol)
  #randomized_vol[...] = vol
  #randomized_vol[...] = 128
  #print("z", randomized_z_coords)
  #print("y", randomized_y_coords)
  #print("x", randomized_x_coords)
  #print("z", flattened_z_coords)
  #print("y", flattened_y_coords)
  #print("x", flattened_x_coords)
  randomized_vol[randomized_z_coords, randomized_y_coords, randomized_x_coords] = vol[flattened_z_coords, flattened_y_coords, flattened_x_coords]
  return randomized_vol

# https://stackoverflow.com/questions/62436299/how-to-lightly-shuffle-a-list-in-python
orderliness = 0.75

def tuplify(x, y):
  return (orderliness * y + np.random.normal(0, 1), x)

def shake(x, y, std_dev=1.0):
  displacements = np.random.normal(0, std_dev, len(x))
  #print(f"{np.min(displacements):.2f} {np.average(np.abs(displacements)):.2f} {np.max(displacements):.2f}", end=' ')
  return np.stack((y + displacements, x), axis=1)

def randomize(vol, mean=0.0, std_dev=1.0):
  print(vol.shape)
  print(std_dev)
  randomized_vol = np.empty_like(vol)

  # Randomization in Z
  values = np.arange(vol.shape[0]).astype(np.int32)
  for y in range(vol.shape[1]):
    print(y, end=' ', flush=True)
    for x in range(vol.shape[2]):
      #pairs = np.array(list(map(tuplify, values, range(len(values)))), dtype=np.int32)
      pairs = shake(values, np.arange(len(values)), std_dev).astype(np.int32)
      pairs = pairs[pairs[:, 0].argsort()]
      randomized_vol[values, y, x] = vol[pairs[:, 1], y , x]

  # Randomization in Y
  values = np.arange(vol.shape[1]).astype(np.int32)
  for z in range(vol.shape[0]):
    print(z, end=' ', flush=True)
    for x in range(vol.shape[2]):
      #pairs = np.array(list(map(tuplify, values, range(len(values)))), dtype=np.int32)
      pairs = shake(values, np.arange(len(values)), std_dev).astype(np.int32)
      pairs = pairs[pairs[:, 0].argsort()]
      randomized_vol[z, values, x] = randomized_vol[z, pairs[:, 1], x]

  # Randomization in X
  #values = np.arange(1, vol.shape[2]+1).astype(np.int32)
  values = np.arange(vol.shape[2]).astype(np.int32)
  for z in range(vol.shape[0]):
    print(z, end=' ', flush=True)
    for y in range(vol.shape[1]):
      #pairs = np.array(list(map(tuplify, values, range(len(values)))), dtype=np.int32)
      pairs = shake(values, np.arange(len(values)), std_dev).astype(np.int32)
      pairs = pairs[pairs[:, 0].argsort()]
      randomized_vol[z, y, values] = randomized_vol[z, y, pairs[:, 1]]

  return randomized_vol

def project_A_to_B(A, B, estimator, block_size, order):
  output_vz, output_vy, output_vx, output_confidence = estimator.calculate_flow(
    A, B,
    start_point=(0, 0, 0),
    total_vol=(A.shape[0], A.shape[1], A.shape[2]),
    sub_volume=block_size,
    overlap=(8, 8, 8),
    threadsperblock=(8, 8, 8)
  )
  print("min flow", np.min(output_vx), np.min(output_vy), np.min(output_vz))
  print("average abs(flow)", np.average(np.abs(output_vx)), np.average(np.abs(output_vy)), np.average(np.abs(output_vz)))
  print("max flow", np.max(output_vx), np.max(output_vy), np.max(output_vz))
  projection = optical_flow_3D.OF3D.generate_inverse_image(A, output_vx, output_vy, output_vz, use_gpu=False, order=order)
  return projection

In [ ]:
block_size = (X.shape[0]//2, X.shape[1]//2, X.shape[2]//2)

In [ ]:
block_size

In [ ]:
A = X
#!mv noisy_vol.mrc 000.mrc
for i in range(N):
    B = randomize(X, std_dev=STD_DEV) # 2.5
    shuffled_X = project_A_to_B(A, B, estimator, block_size, order=0)
    output_filename = f"{i+1:03d}.mrc"
    with mrcfile.new(output_filename, overwrite=True) as mrc:
        mrc.set_data(shuffled_X)
        mrc.data

In [ ]:
!ls -l *.mrc

In [ ]:
shuffled_X.shape

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 4, figsize=(20, 20))

im0 = axes[0].imshow(X[slice_idx][sliice], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice z={slice_idx}')
axes[0].grid(False)

im1 = axes[1].imshow(B[slice_idx][sliice], cmap='gray', origin='lower')
axes[1].set_title(f'Shuffled Slice z={slice_idx}')
axes[1].grid(False)

im2 = axes[2].imshow(shuffled_X[slice_idx][sliice], cmap='gray', origin='lower')
axes[2].set_title(f'Projected Slice z={slice_idx}')
axes[2].grid(False)

im3 = axes[3].imshow((X[slice_idx][sliice] - shuffled_X[slice_idx][sliice] + 128).astype(np.int16) , cmap='gray', origin='lower')
axes[3].set_title(f'original[z] - projected[z]')
axes[3].grid(False)

plt.tight_layout()
plt.show()

# Denoising

In [ ]:
import json

In [ ]:
# %pip install tensorflow

In [ ]:
# %pip install cryoCARE --no-deps

In [ ]:
# %pip install csbdeep

In [ ]:
_ = {
    "even": ["noisy_vol.mrc"],
    "odd": ["shuffled_X.mrc"],
    "mask": [""],
    "patch_shape": [8, 8, 8], # <- Be careful here: in this example the tomogram is very small and the patch shape must be also small
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data_SPRVS",
    "overwrite": "True"
}

_ = {
    "even": ["000.mrc", "002.mrc", "004.mrc", "006.mrc"],
    "odd": ["001.mrc", "003.mrc", "005.mrc", "007.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data_SPRVS",
    "overwrite": "True"
}

def generate_cryocare_config(N):
    #even_tomograms = []
    generated_tomograms = []
    for i in range(N):
        filename = f"{i+1:03d}.mrc"
        # Sort into even or odd lists
        #if i % 2 == 0:
        #    even_tomograms.append(filename)
        #else:
        generated_tomograms.append(filename)
            
    # Construct the final configuration dictionary
    #config = {
    #    "even": even_tomograms,
    #    "odd": odd_tomograms,
    #    "mask": [""],
    #    "patch_shape": [16, 16, 16],
    #    "num_slices": 800,
    #    "split": 0.9,
    #    "tilt_axis": "Y",
    #    "n_normalization_samples": 200,
    #    "path": "./data_SPRVS",
    #    "overwrite": "True"
    #}

    # Construct the final configuration dictionary
    config = {
        "even": ["000.mrc"]*(N),
        "odd": generated_tomograms,
        "mask": [""],
        "patch_shape": [16, 16, 16],
        "num_slices": 800,
        "split": 0.9,
        "tilt_axis": "Y",
        "n_normalization_samples": 200,
        "path": "./data_SPRVS",
        "overwrite": "True"
    }
    
    return config

_ = generate_cryocare_config(N)

with open("train_data_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_data_config__SPRVS.json

In [ ]:
%%bash
/nas/vruiz/envs/OF3D_CUDA/bin/cryoCARE_extract_train_data.py --conf train_data_config__SPRVS.json

In [ ]:
_ = {
  "train_data": "./data_SPRVS",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model_SPRVS",
  "path": "./",
  "gpu_id": [gpu_id]
}
with open("train_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_config__SPRVS.json

In [ ]:
%%bash
/nas/vruiz/envs/OF3D_CUDA/bin/cryoCARE_train.py --conf train_config__SPRVS.json

In [ ]:
_ = {
    "path": "./model_SPRVS.tar.gz",
    "even": ["noisy_vol.mrc"],
    "odd": ["noisy_vol.mrc"],
    "n_tiles": [1,1,1],
    "output": "denoised_vol_SPRVS",
    "overwrite": "True",
    "gpu_id": [1]
}

def generate_cryocare_config(N):
    even_tomograms = []
    odd_tomograms = []
    for i in range(N + 1):
        filename = f"{i:03d}.mrc"
        # Sort into even or odd lists
        if i % 2 == 0:
            even_tomograms.append(filename)
        else:
            odd_tomograms.append(filename)
            
    # Construct the final configuration dictionary
    config = {
        "path": "./model_SPRVS.tar.gz",
        "even": even_tomograms,
        "odd": odd_tomograms,
        "n_tiles": [1,1,1],
        "output": "denoised_vol_SPRVS",
        "overwrite": "True",
        "gpu_id": [1]
    }
    
    return config

_ = generate_cryocare_config(7)

_ = {
    "path": "./model_SPRVS.tar.gz",
    "even": ["000.mrc"],
    "odd": ["000.mrc"],
    "n_tiles": [2,2,2],
    "output": "denoised_vol_SPRVS",
    "overwrite": "True",
    "gpu_id": [1]
}

with open("predict_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat predict_config__SPRVS.json

In [ ]:
%%bash
/nas/vruiz/envs/OF3D_CUDA/bin/cryoCARE_predict.py --conf predict_config__SPRVS.json || true

In [ ]:
!ls -l denoised_vol_SPRVS/*

In [ ]:
REO = mrcfile.open(os.path.expanduser(f"{REO_folder}denoised_vol_REO/000.mrc")).data

sum = np.copy(mrcfile.open(f"denoised_vol_SPRVS/000.mrc").data)
print(sum.dtype)
N = 0
for i in range((N>>1)):
  print(i)
  sum += mrcfile.open(f"denoised_vol_SPRVS/{2*i:03d}.mrc").data
if N>1:
    average = sum/(N>>1)
else:
    average = sum

with mrcfile.new("denoised_vol_SPRVS/average.mrc", overwrite=True) as mrc:
  mrc.set_data(average)
  mrc.data

In [ ]:
#Y = mrcfile.read("denoised_vol_SPRVS/noisy_vol.mrc")
#Y = mrcfile.read("denoised_vol_SPRVS/average.mrc")
Y = mrcfile.read("denoised_vol_SPRVS/000.mrc")

import numpy as np
import mrcfile

file_path = "denoised_vol_SPRVS/noisy_vol.mrc"

with mrcfile.open(file_path, permissive=True) as m:
    nx = m.header.nx
    ny = m.header.ny
    nz = m.header.nz
    nsymbt = m.header.nsymbt

# compute data offset
offset = 1024 + nsymbt

# read raw bytes manually
data = np.fromfile(
    file_path,
    dtype=np.float32,   # <-- force desired dtype
    offset=offset
)

# reshape manually
Y = data.reshape((nz, ny, nx))

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(X[slice_idx][sliice], cmap='gray', origin='lower')
axes.set_title(f'Original Slice Z={slice_idx}')
axes.grid(False)

plt.tight_layout()

plt.savefig('original.png', dpi=75, bbox_inches='tight')

plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[slice_idx][sliice], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[slice_idx][sliice], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-SPRVS Slice Z={slice_idx}, std_dev={STD_DEV}')
axes[1].grid(False)

plt.tight_layout()

plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(Y[slice_idx][sliice], cmap='gray', origin='lower')
axes.set_title(f'N2N-SPRVS Slice Z={slice_idx}, std_dev={STD_DEV}')
axes.grid(False)

plt.tight_layout()

plt.savefig('SPRVS.png', dpi=75, bbox_inches='tight')

plt.show()

slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[slice_idx, 650:850, 800:1000].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[slice_idx, 650:850, 800:1000].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N-Structure Preserving Random Voxel Denoised Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# %pip install --force-reinstall --no-cache-dir "self_fourier_shell_correlation @ git+https://github.com/vicente-gonzalez-ruiz/self_fourier_shell_correlation"
from self_fourier_shell_correlation import fsc_utils as fsc

In [ ]:
# %pip show self_fourier_shell_correlation

In [ ]:
# %pip install  --force-reinstall --no-cache-dir "shuffling @ git+https://github.com/vicente-gonzalez-ruiz/shuffling"

In [ ]:
!ls -l

import numpy as np
import mrcfile

file_path = "denoised_vol_REO/noisy_vol.mrc"

with mrcfile.open(file_path, permissive=True) as m:
    nx = m.header.nx
    ny = m.header.ny
    nz = m.header.nz
    nsymbt = m.header.nsymbt

# compute data offset
offset = 1024 + nsymbt

# read raw bytes manually
data = np.fromfile(
    file_path,
    dtype=np.float32,   # <-- force desired dtype
    offset=offset
)

# reshape manually
REO = data.reshape((nz, ny, nx))

In [ ]:
list_fsc_values__X = []
list_fsc_values__REO = []
list_fsc_values__SPRVS = []
for i in range(X.shape[0]-3):
    print(i, '/', X.shape[0])
    spatial_freqs, fsc_values__X = fsc.get_SFRC_curve__subsampled_chessboard(X[i])
    spatial_freqs, fsc_values__REO = fsc.get_SFRC_curve__subsampled_chessboard(REO[i])
    spatial_freqs, fsc_values__SPRVS = fsc.get_SFRC_curve__subsampled_chessboard(Y[i])
    list_fsc_values__X.append(fsc_values__X)
    list_fsc_values__REO.append(fsc_values__REO)
    list_fsc_values__SPRVS.append(fsc_values__SPRVS)

In [ ]:
avg_fsc_values__X = np.mean(list_fsc_values__X, axis=0)
avg_fsc_values__REO = np.mean(list_fsc_values__REO, axis=0)
avg_fsc_values__SPRVS = np.mean(list_fsc_values__SPRVS, axis=0)

In [ ]:
plt.title(f"{Path.cwd().parts[-2:]}, std_dev={STD_DEV}, ss={SPATIAL_SIZE}, sk={SIGMA_K}, fs={FILTER_SIZE}")
plt.xlabel("Normalized Spatial Frequency (cycles/pixel)")
plt.ylabel("Average Self Fourier Ring Correlation")
plt.plot(spatial_freqs, avg_fsc_values__X, label="Noisy", color="blue")
plt.plot(spatial_freqs, avg_fsc_values__REO, label="N2N-REO", color="red")
plt.plot(spatial_freqs, avg_fsc_values__SPRVS, label="N2N-SPRVS", color="green")
plt.legend(loc='lower left')

In [ ]:
import scipy.stats

In [ ]:
def PCC(original, denoised):
    return scipy.stats.pearsonr(original.flatten(), denoised.flatten())[0]

In [ ]:
print(f"PCC={PCC(X, Y):.3f}")

In [ ]:
import skimage.metrics

In [ ]:
def PSNR(original, denoised):
    return skimage.metrics.peak_signal_noise_ratio(original, denoised, data_range=original.max()-original.min())

In [ ]:
print(f"PSNR={PSNR(X, Y):.3f}")

In [ ]:
def SSIM(original, denoised):
    return skimage.metrics.structural_similarity(original, denoised, data_range=original.max() - original.min())

In [ ]:
print(f"SSIM={SSIM(X, Y):.3f}")